# 1. Create your first Agent Framework agent

Start here: create a Python agent, call a model through an existing Microsoft Foundry project using Microsoft Entra sign-in, and keep a conversation in memory.

**Agent Framework** runs the agent: instructions, model calls, conversation state, and (later) tools. **Agent 365** adds enterprise identity, registration, governance, Microsoft 365 tools, and observability. Creating an `Agent` here does **not** register it with Agent 365 or deploy it to Azure.

This notebook runs locally. Only the prompts are sent to your Azure model deployment, and model usage may incur charges.

## Prerequisites

- Python 3.10 or later and a Python notebook kernel in VS Code.
- An existing Microsoft Foundry project with a model deployment that supports the Responses API.
- Its **project endpoint** (for example, `https://YOUR-RESOURCE.services.ai.azure.com/api/projects/YOUR-PROJECT`) and **model deployment name**.
- Permission to use the project and call its model, normally the **Azure AI User** role at the appropriate project/resource scope.

In a PowerShell terminal, sign in to the tenant containing your model:

```powershell
az login
```

For a specific tenant, use `az login --tenant YOUR-TENANT-ID`. `AzureCliCredential` below uses this sign-in; no API key is needed.

## Set up the Python environment

`agent-framework-foundry` includes the core framework and Microsoft Foundry integration. `requirements.txt` pins the framework packages to the verified versions.

From this project's directory in PowerShell:

```powershell
uv venv .venv --python 3.12
uv pip install --python .venv\Scripts\python.exe -r requirements.txt
```

In VS Code, choose **Select Kernel > Python Environments** and select this project's `.venv`. **Restart the kernel after installing or updating dependencies**, then rerun from the imports to avoid mixing previously imported modules with upgraded packages. No shell activation is required. The managed-device uv configuration uses the Microsoft package proxy.

Run the following cells in order using the `.venv` kernel.

In [1]:
import os

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

## Configure your model

Fill in `.env` in this project's directory:

```dotenv
FOUNDRY_PROJECT_ENDPOINT=https://YOUR-RESOURCE.services.ai.azure.com/api/projects/YOUR-PROJECT
FOUNDRY_MODEL=YOUR-DEPLOYMENT-NAME
```

The deployment name is the name you assigned in Azure, not necessarily the underlying model name. Run the notebook with the project directory as its working directory.

`.env` is ignored by Git. Authentication still uses your Azure CLI sign-in, not an API key. Existing environment variables take precedence over `.env`; if you edit the file, restart the kernel before rerunning.

In [2]:
load_dotenv(".env")

project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "").strip().rstrip("/")
deployment_name = os.getenv("FOUNDRY_MODEL", "").strip()

if not project_endpoint.startswith("https://") or not project_endpoint.partition("/api/projects/")[2]:
    raise ValueError("Set FOUNDRY_PROJECT_ENDPOINT in .env to your HTTPS Foundry project endpoint, including /api/projects/YOUR-PROJECT.")
if not deployment_name:
    raise ValueError("Set FOUNDRY_MODEL in .env to your model deployment name.")

## Create the agent

The **client** connects to the model. The **instructions** define the agent's behavior. The **name** is a local label, not an Agent 365 identity.

`FoundryChatClient` uses the project's Responses API. Creating these objects does not call the model or publish an agent into the Foundry catalog.

In [3]:
credential = AzureCliCredential()

client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=deployment_name,
    credential=credential,
)

agent = Agent(
    client=client,
    name="LearningAssistant",
    instructions=(
        "You are a patient technical tutor. Explain concepts in plain language, "
        "keep answers short, and give one concrete example. "
        "Be explicit when you are uncertain; do not invent product capabilities."
    ),
)

print(f"Created agent: {agent.name}")

Created agent: LearningAssistant


## Send a first prompt

This cell makes the first model call. A call without a session is stateless: it does not retain this exchange for future calls.

Jupyter supports top-level `await`, so we do not need `asyncio.run()`.

In [4]:
response = await agent.run("What is an AI agent, and how is it different from a single model call?")
print(response.text)

An **AI agent** is a system that works toward a goal by deciding what to do next. It may call a model repeatedly, use tools, check results, and adjust its plan.

A **single model call** takes an input and returns an answer once. It usually doesn’t take further actions unless another part of the system asks it to.

**Example:** Ask a model, “What’s a good flight to Boston?” and it might suggest options. An agent could search flight listings, compare them against your preferences, and—if permitted—book one.

The term “agent” is used in different ways, but the key idea is a loop of deciding and acting, rather than one response.


## Keep a conversation with a session

Pass the same session to both calls so the follow-up includes the earlier conversation. This session is in local memory, not an Agent 365 registration or durable hosted session. Restarting the kernel loses it.

The first stateless prompt above is not part of this new session.

In [5]:
session = agent.create_session()

response = await agent.run(
    "I am learning to build a helpdesk agent. Suggest one simple tool it could use.",
    session=session,
)
print(response.text)

A simple tool is **`get_ticket_status(ticket_id)`**. It looks up a ticket’s current status so the agent can answer, for example: “Ticket 4821 is waiting for a support specialist.”


In [6]:
response = await agent.run(
    "What inputs would that tool need? Refer to your previous suggestion.",
    session=session,
)
print(response.text)

For the `get_ticket_status(ticket_id)` tool, the main input is:

- **`ticket_id`** — the identifier for the ticket, such as `4821`.

The tool can use that ID to look up the ticket’s current status.


## Close connections

Run this when finished. To call the agent again afterward, rerun the agent-creation cell first.

In [7]:
await client.close()
await credential.close()

## What you built

A local agent with instructions, an authenticated Foundry project client, and a multi-turn conversation. No tools, hosting, published Foundry agent, Agent 365 identity, or Agent 365 telemetry have been added yet.

The natural next lesson is to add a small Python tool, then introduce Agent 365 registration and observability separately.

### Troubleshooting

- **Import error:** run the uv install command above, select the project's `.venv` kernel, and rerun from the imports.
- **Authentication error:** run `az login` in the correct tenant.
- **403:** check model-access permissions and resource network restrictions.
- **404 / deployment not found:** check the project endpoint and deployment name, and confirm Responses API support.

### References

- [Microsoft Agent Framework](https://github.com/microsoft/agent-framework)
- [Python Microsoft Foundry integration](https://github.com/microsoft/agent-framework/tree/main/python/packages/foundry)
- [Agent 365 overview](https://learn.microsoft.com/en-us/microsoft-agent-365/overview)